# Week 4 — Classification

**Focus:** How do scores become trustworthy probabilities and defensible policies?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week04/week4_demos.ipynb)

## What this notebook is about

A classification score is not automatically a trustworthy probability, and a probability is not automatically a good decision. This notebook separates those three objects: **score, probability, and policy**.

The demonstrations compare probability models with feasible base rates, separate ranking from probability quality, move a threshold while holding the model fixed, freeze a cost-sensitive policy before test, draw overlapping label intervals, and separate inner selection from outer assessment. Market direction and credit are used as examples, but the logic applies to fraud, default, approval, intervention, and other financial events.

By the end, you should be able to:

- distinguish discrimination from probability quality;
- explain why a threshold belongs to a decision policy rather than to the model alone;
- select a policy using stated costs or benefits and then freeze it;
- identify when overlapping labels cross a validation boundary; and
- keep model selection separate from final assessment.

The examples are controlled comparisons. Their numerical scores illustrate the evaluation logic; they do not establish a deployable credit or trading system.

### Course context

For a concise review and the optional textbook route, see [Week 4 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week04/week4_summary.md) and [Week 4 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week04/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — Direction: compare the probability model with a feasible baseline — <a href="#Demo-1-%E2%80%94-Direction%3A-compare-the-probability-model-with-a-feasible-baseline">Jupyter</a> · <a href="#scrollTo=md-0e7399164dd4">Colab</a></li>
<li>Real-data companion — next-day market direction against its base rate — <a href="#Real-data-companion-%E2%80%94-next-day-market-direction-against-its-base-rate">Jupyter</a> · <a href="#scrollTo=md-2052b6c7c558">Colab</a></li>
<li>Demo 2 — Credit: ranking and probability quality are separate — <a href="#Demo-2-%E2%80%94-Credit%3A-ranking-and-probability-quality-are-separate">Jupyter</a> · <a href="#scrollTo=md-05d4d4cdda80">Colab</a></li>
<li>Demo 3 — One probability model, several threshold policies — <a href="#Demo-3-%E2%80%94-One-probability-model,-several-threshold-policies">Jupyter</a> · <a href="#scrollTo=md-d64568165f9c">Colab</a></li>
<li>Demo 4 — Select a cost-sensitive threshold, then freeze it for test — <a href="#Demo-4-%E2%80%94-Select-a-cost-sensitive-threshold,-then-freeze-it-for-test">Jupyter</a> · <a href="#scrollTo=md-5e314cd1c568">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 4b — Similar model files, different search histories — <a href="#Demo-4b-%E2%80%94-Similar-model-files,-different-search-histories">Jupyter</a> · <a href="#scrollTo=md-c562c7ca8a28">Colab</a></li>
<li>Demo 5 — Draw label intervals and compare validation clocks — <a href="#Demo-5-%E2%80%94-Draw-label-intervals-and-compare-validation-clocks">Jupyter</a> · <a href="#scrollTo=md-49d30e40fd94">Colab</a></li>
<li>Demo 6 — Three production clocks — <a href="#Demo-6-%E2%80%94-Three-production-clocks">Jupyter</a> · <a href="#scrollTo=md-c0fc1f4f201e">Colab</a></li>
<li>Demo 7 — Nested selection exposes the difference between choosing and assessing — <a href="#Demo-7-%E2%80%94-Nested-selection-exposes-the-difference-between-choosing-and-assessing">Jupyter</a> · <a href="#scrollTo=md-884ff748fdfe">Colab</a></li>
</ul>

## Your Week 4 practice research record

Complete the editable entry immediately below each demonstration. **Do not write in this overview.** These entries stay in the weekly notebook and are not a separate graded submission.

### The five lecture breaks

1. **Grade probabilities — Demos 1–2:** compare each model with its frozen training-rate baseline; then use the overconfident credit forecast to separate ranking from probability quality.
2. **Move the threshold — Demo 3:** hold probabilities fixed, record three operating points, and define what *best* means before choosing one.
3. **Freeze the policy — Demo 4:** select the threshold on validation, assess the model and policies on test, and assign each result to the model or model-policy pair.
4. **Audit the boundary — Demo 5:** store label intervals, identify the exact crossing rows, and compare shuffled, forward, and purged-forward questions.
5. **Build the production split — Demos 6–7:** document the protected unit for three data shapes, then keep inner selection separate from outer assessment and its baseline.

The real-market direction companion after Demo 1 is optional and has its own entry. For every result, state a narrow claim, its owner—model or policy—and one limitation.

### Imports and shared helpers

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url


def mse(a, b):
    """Mean squared error, used by the forward-validation demos."""
    return np.mean((np.asarray(a) - np.asarray(b)) ** 2)


def sigmoid(z):
    """Numerically stable enough for the ranges used in these demonstrations."""
    return 1.0 / (1.0 + np.exp(-z))


def expected_calibration_error(y, p, n_bins=10):
    """Frequency-weighted absolute reliability gap."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bins = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    total = len(y)
    ece = 0.0
    rows = []
    for b in range(n_bins):
        mask = bins == b
        if not mask.any():
            continue
        mean_p = float(np.mean(p[mask]))
        event_rate = float(np.mean(y[mask]))
        weight = int(mask.sum()) / total
        ece += weight * abs(mean_p - event_rate)
        rows.append((int(mask.sum()), mean_p, event_rate))
    return ece, pd.DataFrame(rows, columns=["n", "mean_probability", "event_rate"])


def classification_row(y, p, threshold):
    """Metrics belonging to one frozen threshold policy."""
    pred = (p >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "review_rate": pred.mean(),
    }


def build_credit_experiment():
    """Build the deterministic three-cohort credit experiment used in Demos 2–4."""
    rng = np.random.default_rng(2)
    n_obs = 9000
    dates = pd.bdate_range("1990-01-02", periods=n_obs)
    time = np.arange(n_obs)
    features = rng.standard_normal((n_obs, 4))

    # The final cohort has a higher base rate. Feature relationships stay fixed,
    # while the probability level changes.
    cohort_shift = 0.35 * (time >= 6750)
    logit = (
        -3.0
        + 1.0 * features[:, 0]
        + 0.6 * features[:, 1]
        - 0.4 * features[:, 2]
        + 0.2 * features[:, 3]
        + cohort_shift
    )
    outcome = (rng.uniform(size=n_obs) < sigmoid(logit)).astype(int)
    train = np.arange(0, 4500)
    validation = np.arange(4500, 6750)
    test = np.arange(6750, 9000)

    model = LogisticRegression(C=np.inf, max_iter=2000).fit(
        features[train], outcome[train]
    )
    p_validation = model.predict_proba(features[validation])[:, 1]
    p_test = model.predict_proba(features[test])[:, 1]
    base_rate = outcome[train].mean()
    p_baseline = np.repeat(base_rate, len(test))
    return {
        "dates": dates,
        "X": features,
        "y": outcome,
        "train": train,
        "validation": validation,
        "test": test,
        "model": model,
        "p_validation": p_validation,
        "p_test": p_test,
        "base_rate": base_rate,
        "p_baseline": p_baseline,
    }

### Demo 1 — Direction: compare the probability model with a feasible baseline

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
rng = np.random.default_rng(1)
T = 5000
direction_dates = pd.bdate_range("2000-01-03", periods=T)
direction_x = rng.standard_normal(T)
direction_p = sigmoid(0.10 * direction_x)
direction_y = (rng.uniform(size=T) < direction_p).astype(int)
direction_split = 3000

X_train = direction_x[:direction_split, None]
X_test = direction_x[direction_split:, None]
y_train = direction_y[:direction_split]
y_test = direction_y[direction_split:]

direction_model = LogisticRegression(C=np.inf, max_iter=2000).fit(X_train, y_train)
p_test = direction_model.predict_proba(X_test)[:, 1]
p_baseline = np.repeat(y_train.mean(), len(y_test))

direction_results = pd.DataFrame(
    [
        {
            "model": "training base rate",
            "AUC": 0.5,
            "Brier": brier_score_loss(y_test, p_baseline),
            "log_loss": log_loss(y_test, p_baseline),
            "accuracy": accuracy_score(y_test, p_baseline >= 0.5),
        },
        {
            "model": "logistic",
            "AUC": roc_auc_score(y_test, p_test),
            "Brier": brier_score_loss(y_test, p_test),
            "log_loss": log_loss(y_test, p_test),
            "accuracy": accuracy_score(y_test, p_test >= 0.5),
        },
    ]
)
print("Direction: one later test block")
print(f"Training positive-day rate: {y_train.mean():.3f}")
print(f"Train: {direction_dates[0].date()} through {direction_dates[direction_split - 1].date()}")
print(f"Test:  {direction_dates[direction_split].date()} through {direction_dates[-1].date()}")
print(direction_results.round(4).to_string(index=False))
# Expected: the logistic barely clears the base rate (AUC ~0.54, log loss ~0.693
# versus ~0.694). A very small edge in one simulated later block is the honest result.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| known truth and target | `[event, horizon, and planted structure]` |
| training-rate baseline | `[frozen probability and fit population]` |
| boundary | `[train and test dates]` |
| baseline evidence | `[AUC, Brier, log loss, and accuracy]` |
| logistic evidence | `[AUC, Brier, log loss, and accuracy]` |
| narrow claim | `[what improved in this later block]` |
| limitation | `[what this experiment cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed direction entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| known truth and target | synthetic next-session direction; event probability is `sigmoid(0.10 × x)`, so directional structure is deliberately weak |
| training-rate baseline | freeze the training positive-day rate **0.479** for every test forecast |
| boundary | train 2000-01-03–2011-07-01; test 2011-07-04–2019-03-01 |
| baseline evidence | AUC **0.5000**, Brier **0.2505**, log loss **0.6942**, accuracy **0.4980** |
| logistic evidence | AUC **0.5367**, Brier **0.2498**, log loss **0.6928**, accuracy **0.5145** |
| narrow claim | logistic improved this one later block slightly on both ranking and proper probability loss |
| limitation | one independent-row simulation and one split do not establish stable market-direction predictability; accuracy depends on the 0.5 policy threshold |

The model competes with a baseline that could actually have been issued before
the test outcomes arrived.

</details>

### Real-data companion — next-day market direction against its base rate

> **Optional historical direction contest:** Deck A, after recording segment S1. This uses the same kind of binary target as Demo 1, but a different feature set and a historical population. Record its own clock and frozen base-rate benchmark. Treat a small metric difference as evidence from this one later period, not as a stable edge.

In [ ]:
ff_direction_raw, ff_direction_source = course_csv("datasets/famafrench/ff_factors_daily.csv")
ff_direction_raw["date"] = pd.to_datetime(
    ff_direction_raw["date"].astype(str), format="%Y%m%d"
)
ff_direction = ff_direction_raw.set_index("date").sort_index().loc["1990":]
market_direction_return = (ff_direction["Mkt-RF"] + ff_direction["RF"]) / 100.0
direction_frame = pd.DataFrame({
    "return_t": market_direction_return,
    "mom5": market_direction_return.rolling(5).mean(),
    "mom21": market_direction_return.rolling(21).mean(),
    "vol21": market_direction_return.rolling(21).std(),
    "up_next": (market_direction_return.shift(-1) > 0).where(
        market_direction_return.shift(-1).notna()
    ),
}).dropna()
real_direction_cut = int(0.7 * len(direction_frame))
real_direction_train = direction_frame.iloc[:real_direction_cut]
real_direction_test = direction_frame.iloc[real_direction_cut:]
real_direction_columns = ["return_t", "mom5", "mom21", "vol21"]
real_direction_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
real_direction_model.fit(
    real_direction_train[real_direction_columns], real_direction_train["up_next"].astype(int)
)
real_direction_probability = real_direction_model.predict_proba(
    real_direction_test[real_direction_columns]
)[:, 1]
real_direction_y_test = real_direction_test["up_next"].astype(int).to_numpy()
real_direction_baseline = np.repeat(real_direction_train["up_next"].mean(), len(real_direction_test))
real_direction_results = pd.DataFrame([
    {
        "model": "training base rate",
        "AUC": 0.5,
        "Brier": brier_score_loss(real_direction_y_test, real_direction_baseline),
        "log_loss": log_loss(real_direction_y_test, real_direction_baseline),
        "accuracy": accuracy_score(real_direction_y_test, real_direction_baseline >= 0.5),
    },
    {
        "model": "logistic",
        "AUC": roc_auc_score(real_direction_y_test, real_direction_probability),
        "Brier": brier_score_loss(real_direction_y_test, real_direction_probability),
        "log_loss": log_loss(real_direction_y_test, real_direction_probability),
        "accuracy": accuracy_score(real_direction_y_test, real_direction_probability >= 0.5),
    },
])
print("Source: Kenneth R. French Data Library, bundled daily factors")
print(f"File: {ff_direction_source}")
print(f"Train: {real_direction_train.index.min().date()} through {real_direction_train.index.max().date()}")
print(f"Test:  {real_direction_test.index.min().date()} through {real_direction_test.index.max().date()}")
print("Decision: after close t; target: whether the market return on t+1 is positive")
print(real_direction_results.round(4).to_string(index=False))
print("This is one frozen historical contest. It does not turn a small metric difference into a stable edge.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| source and clock | `[source, decision time, and target interval]` |
| boundary | `[train and test dates]` |
| feasible baseline | `[probability and freeze rule]` |
| baseline evidence | `[AUC, Brier, log loss, and accuracy]` |
| logistic evidence | `[AUC, Brier, log loss, and accuracy]` |
| narrow claim | `[what this historical contest supports]` |
| limitation | `[what it does not prove]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed optional market-direction entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| source and clock | Kenneth R. French daily US market data; decide after close *t* and predict whether return *t*+1 is positive |
| boundary | train 1990-01-30–2015-06-08; test 2015-06-09–2026-04-29 |
| feasible baseline | training positive-day rate, frozen through test |
| baseline evidence | AUC **0.5000**, Brier **0.2480**, log loss **0.6891**, accuracy **0.5451** |
| logistic evidence | AUC **0.4969**, Brier **0.2482**, log loss **0.6895**, accuracy **0.5436** |
| narrow claim | in this one historical contest, the four-feature logistic model did not improve the frozen base-rate forecast |
| limitation | one feature set, split, horizon, and market series do not prove that direction is universally unpredictable |

This comparison is useful precisely because the richer model does not receive
credit merely for being fitted.

</details>

### Demo 2 — Credit: ranking and probability quality are separate

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
credit = build_credit_experiment()
credit_dates = credit["dates"]
credit_y = credit["y"]
credit_train = credit["train"]
credit_validation = credit["validation"]
credit_test = credit["test"]
credit_p_validation = credit["p_validation"]
credit_p_test = credit["p_test"]
credit_base_rate = credit["base_rate"]
credit_p_baseline = credit["p_baseline"]

# A monotone transformation preserves every rank and deliberately damages
# calibration.
eps = 1e-8
test_score = np.log(
    np.clip(credit_p_test, eps, 1 - eps)
    / np.clip(1 - credit_p_test, eps, 1 - eps)
)
credit_p_overconfident = sigmoid(2.5 * test_score)

credit_probability_results = pd.DataFrame(
    [
        {
            "forecast": "training base rate",
            "AUC": 0.5,
            "Brier": brier_score_loss(credit_y[credit_test], credit_p_baseline),
            "log_loss": log_loss(credit_y[credit_test], credit_p_baseline),
        },
        {
            "forecast": "logistic",
            "AUC": roc_auc_score(credit_y[credit_test], credit_p_test),
            "Brier": brier_score_loss(credit_y[credit_test], credit_p_test),
            "log_loss": log_loss(credit_y[credit_test], credit_p_test),
        },
        {
            "forecast": "same ranks, overconfident",
            "AUC": roc_auc_score(credit_y[credit_test], credit_p_overconfident),
            "Brier": brier_score_loss(credit_y[credit_test], credit_p_overconfident),
            "log_loss": log_loss(credit_y[credit_test], credit_p_overconfident),
        },
    ]
)
print("\nCredit: later test cohort")
print(f"Training default rate: {credit_base_rate:.3f}")
print(f"Test default rate:     {credit_y[credit_test].mean():.3f}")
print(f"Train: {credit_dates[credit_train[0]].date()} through {credit_dates[credit_train[-1]].date()}")
print(f"Validation: {credit_dates[credit_validation[0]].date()} through {credit_dates[credit_validation[-1]].date()}")
print(f"Test: {credit_dates[credit_test[0]].date()} through {credit_dates[credit_test[-1]].date()}")
print(credit_probability_results.round(4).to_string(index=False))
# Expected: AUC ~0.75 for both logistic forecasts -- identical ranking -- while
# Brier and log loss expose the overconfident probability scale.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| target and cohorts | `[event plus train, validation, and test dates]` |
| population change | `[training and test prevalence plus planted shift]` |
| training-rate baseline | `[AUC, Brier, and log loss]` |
| logistic forecast | `[AUC, Brier, and log loss]` |
| same ranks, overconfident | `[AUC, Brier, and log loss]` |
| narrow claim | `[what AUC can and cannot establish]` |
| limitation | `[what the simulation leaves out]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed credit-probability entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| target and cohorts | synthetic default event; train 1990-01-02–2007-04-02, validation 2007-04-03–2015-11-16, test 2015-11-17–2024-07-01 |
| population change | training default rate **0.084**; test default rate **0.104** after a planted cohort-level log-odds shift |
| training-rate baseline | AUC **0.5000**, Brier **0.0932**, log loss **0.3352** |
| logistic forecast | AUC **0.7517**, Brier **0.0833**, log loss **0.2954** |
| same ranks, overconfident | AUC remains **0.7517**, while Brier worsens to **0.0934** and log loss to **0.5422** |
| narrow claim | AUC certifies an ordering property here; it cannot tell the pricing desk whether the probability scale is trustworthy |
| limitation | the logistic data-generating relation is stable except for one planted cohort shift; real credit populations can change in many additional ways |

The monotone transformation damages probability quality without changing a
single rank. That is the controlled contrast.

</details>

### Demo 3 — One probability model, several threshold policies

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
credit = build_credit_experiment()
credit_y = credit["y"]
credit_validation = credit["validation"]
credit_p_validation = credit["p_validation"]
validation_event_rate = credit_y[credit_validation].mean()
majority_accuracy = max(validation_event_rate, 1 - validation_event_rate)
threshold_rows = [
    classification_row(credit_y[credit_validation], credit_p_validation, threshold)
    for threshold in [0.50, 0.20, 0.10]
]
threshold_table = pd.DataFrame(threshold_rows)
print("\nValidation policies from one fitted probability model")
print(f"Validation event rate: {validation_event_rate:.4f}; majority-policy accuracy: {majority_accuracy:.4f}")
print(threshold_table.round(4).to_string(index=False))

for threshold in [0.50, 0.10]:
    pred = (credit_p_validation >= threshold).astype(int)
    print(f"\nValidation confusion matrix at threshold {threshold:.2f}")
    print(confusion_matrix(credit_y[credit_validation], pred))

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| fixed object | `[probability vector and what does not change]` |
| population reference | `[event rate and majority-policy accuracy]` |
| threshold 0.50 | `[accuracy, precision, recall, and review rate]` |
| threshold 0.20 | `[accuracy, precision, recall, and review rate]` |
| threshold 0.10 | `[accuracy, precision, recall, and review rate]` |
| declared meaning of best | `[objective, constraint, and selected threshold]` |
| narrow claim | `[what moving the threshold changes]` |
| limitation | `[which alternative objective could change the choice]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed threshold-policy entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| fixed object | one validation probability vector from the credit logistic model; probabilities and AUC do not change |
| population reference | validation event rate **0.0858**; majority-policy accuracy **0.9142** |
| threshold 0.50 | accuracy **0.9173**, precision **0.6842**, recall **0.0674**, review rate **0.0084** |
| threshold 0.20 | accuracy **0.8804**, precision **0.3190**, recall **0.3472**, review rate **0.0933** |
| threshold 0.10 | accuracy **0.7676**, precision **0.2032**, recall **0.5855**, review rate **0.2471** |
| declared meaning of best | if review capacity is at most 10% and recall is preferred within that constraint, choose **0.20**; it reviews 9.33% and catches 34.72% of defaults |
| narrow claim | moving the threshold changes the queue and error tradeoff, not the fitted probabilities or ranking |
| limitation | the recommendation depends on the stated capacity objective; different false-negative costs or capacity would justify another threshold |

There is no model-free best threshold. The objective completes the policy.

</details>

### Demo 4 — Select a cost-sensitive threshold, then freeze it for test

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
COST_FN = 10.0
COST_FP = 1.0
threshold_grid = np.linspace(0.01, 0.60, 120)

credit = build_credit_experiment()
credit_dates = credit["dates"]
credit_y = credit["y"]
credit_validation = credit["validation"]
credit_test = credit["test"]
credit_p_validation = credit["p_validation"]
credit_p_test = credit["p_test"]
credit_p_baseline = credit["p_baseline"]


def policy_cost(y, p, threshold):
    pred = p >= threshold
    false_negative = np.sum((pred == 0) & (y == 1))
    false_positive = np.sum((pred == 1) & (y == 0))
    return (COST_FN * false_negative + COST_FP * false_positive) / len(y)


validation_costs = np.array(
    [
        policy_cost(credit_y[credit_validation], credit_p_validation, threshold)
        for threshold in threshold_grid
    ]
)
selected_threshold = float(threshold_grid[np.argmin(validation_costs)])
analytic_threshold = COST_FP / (COST_FP + COST_FN)

test_ece, test_reliability = expected_calibration_error(
    credit_y[credit_test], credit_p_test
)
baseline_ece, _ = expected_calibration_error(
    credit_y[credit_test], credit_p_baseline
)
test_model_table = pd.DataFrame([
    {
        "forecast": "training base rate",
        "AUC": 0.5,
        "Brier": brier_score_loss(credit_y[credit_test], credit_p_baseline),
        "log_loss": log_loss(credit_y[credit_test], credit_p_baseline),
        "ECE": baseline_ece,
    },
    {
        "forecast": "logistic",
        "AUC": roc_auc_score(credit_y[credit_test], credit_p_test),
        "Brier": brier_score_loss(credit_y[credit_test], credit_p_test),
        "log_loss": log_loss(credit_y[credit_test], credit_p_test),
        "ECE": test_ece,
    },
])
policy_results = []
for name, threshold in [
    ("validation-selected", selected_threshold),
    ("analytic cost ratio", analytic_threshold),
    ("library default", 0.50),
]:
    row = classification_row(credit_y[credit_test], credit_p_test, threshold)
    row["policy"] = name
    row["cost_per_case"] = policy_cost(
        credit_y[credit_test], credit_p_test, threshold
    )
    policy_results.append(row)

print("\nFrozen probability model on the test cohort")
print(f"Validation: {credit_dates[credit_validation[0]].date()} through {credit_dates[credit_validation[-1]].date()}")
print(f"Test:       {credit_dates[credit_test[0]].date()} through {credit_dates[credit_test[-1]].date()}")
print("Model-level test comparison")
print(test_model_table.round(4).to_string(index=False))
print(f"Validation-selected threshold: {selected_threshold:.3f}")
print(f"Analytic cost-ratio threshold: {analytic_threshold:.3f}")
print("Policy-level test comparison")
print(pd.DataFrame(policy_results).round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(threshold_grid, validation_costs, color="tab:blue")
axes[0].axvline(selected_threshold, color="black", ls="--", label="validation choice")
axes[0].axvline(analytic_threshold, color="0.5", ls=":", label="cost-ratio threshold")
axes[0].set(xlabel="threshold", ylabel="validation cost per case")
axes[0].legend()

axes[1].plot([0, 1], [0, 1], color="0.5", ls="--")
axes[1].scatter(
    test_reliability["mean_probability"],
    test_reliability["event_rate"],
    s=np.maximum(test_reliability["n"], 20) / 3,
    color="tab:blue",
)
axes[1].set(
    xlabel="mean forecast probability",
    ylabel="observed event rate",
    xlim=(0, 0.6),
    ylim=(0, 0.6),
)
fig.tight_layout()
plt.show()

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| declared costs | `[cost of a missed event and cost of a false alarm]` |
| selection and assessment boundary | `[validation and test dates, plus what was frozen]` |
| thresholds | `[validation-selected, analytic, and default thresholds]` |
| model-level test evidence | `[baseline and logistic AUC, Brier, log loss, and ECE]` |
| selected-policy evidence | `[accuracy, precision, recall, review rate, and average cost]` |
| alternative-policy evidence | `[same fields for the analytic and 0.50 policies]` |
| ownership | `[what belongs to the probability model and what belongs to the policy]` |
| narrow claim | `[what the untouched test comparison supports]` |
| limitation | `[what the selected policy still does not establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed frozen-policy entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| declared costs | false negative **10** units; false positive **1** unit; cost per case is evaluated on mature outcomes |
| selection and assessment boundary | validation 2007-04-03–2015-11-16 selects the threshold; test 2015-11-17–2024-07-01 assesses it once |
| thresholds | validation-selected **0.084**; analytic cost-ratio **0.091**; library default **0.500** |
| model-level test evidence | logistic: AUC **0.7517**, Brier **0.0833**, log loss **0.2954**, ECE **0.0262**; training-rate baseline: **0.5000**, **0.0932**, **0.3352**, **0.0196** |
| selected-policy evidence | review rate **0.3067**, recall **0.6180**, cost per case **0.6382** |
| alternative-policy evidence | analytic threshold: review **0.2836**, cost **0.6640**; 0.5 threshold: review **0.0067**, cost **0.9884** |
| ownership | AUC, Brier, log loss, and ECE describe the model on test; review rate and realized cost describe the frozen model-policy pair |
| narrow claim | the validation-selected policy had the lowest test cost of the three prespecified policies in this cohort |
| limitation | ECE depends on the binning rule, costs are stylized, and the final cohort has a planted base-rate shift; test results must not retune the threshold |

The lower baseline ECE does not make the constant forecast the better model:
the proper losses and ranking answer different questions from this binned
calibration summary.

</details>

### Demo 4b — Similar model files, different search histories

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Two researchers produce the same kind of model file: five ridge coefficients
# fitted on the same rows. One specified the candidate set before opening later
# outcomes; the other searched the later period for a favorable subset. Neither
# file records the search that produced it.
rng = np.random.default_rng(404)
n_rows, n_cols = 900, 40
audit_X = rng.standard_normal((n_rows, n_cols))
audit_y = rng.standard_normal(n_rows)          # no signal: nothing is learnable

train = np.arange(0, 500)
later = np.arange(500, 620)                    # the period both researchers score
fresh = np.arange(620, n_rows)                 # a period neither one ever saw

# 200 candidates against a 120-row scoring window: enough search pressure that
# the best-looking subset is chosen by noise rather than by structure.
candidates = [rng.choice(n_cols, size=5, replace=False) for _ in range(200)]

def fit_and_score(columns, fit_rows, score_rows):
    model = Ridge(alpha=1.0).fit(audit_X[fit_rows][:, columns], audit_y[fit_rows])
    return model, mse(audit_y[score_rows], model.predict(audit_X[score_rows][:, columns]))

# Researcher A: one candidate named in advance, then scored once on `later`.
prespecified = candidates[0]
model_a, reported_a = fit_and_score(prespecified, train, later)

# Researcher B: score every candidate on `later`, keep the best, report that score.
scored = [fit_and_score(cols, train, later)[1] for cols in candidates]
best = int(np.argmin(scored))
selected = candidates[best]
model_b, reported_b = fit_and_score(selected, train, later)

# Now ask the question neither reported score answers: how does each do on a
# period that was never consulted?
_, honest_a = fit_and_score(prespecified, train, fresh)
_, honest_b = fit_and_score(selected, train, fresh)

audit = pd.DataFrame([
    {"researcher": "A (prespecified)", "reported_loss": reported_a, "fresh_loss": honest_a},
    {"researcher": "B (searched 200)",  "reported_loss": reported_b, "fresh_loss": honest_b},
])
print("\nSimilar model files, different search histories")
print(audit.round(4).to_string(index=False))
gap_reported = reported_a - reported_b
gap_fresh = honest_a - honest_b
print(f"\nOn the period B searched, B's loss is lower than A's by {gap_reported:.4f}.")
print(
    f"On the untouched period B is "
    f"{'still ahead by ' + format(gap_fresh, '.4f') if gap_fresh > 0 else 'BEHIND by ' + format(-gap_fresh, '.4f')}."
)
print("Both models are five ridge coefficients fitted on the same 500 rows.")
print("Nothing in either saved file records how many candidates were examined.")
# Expected: B's reported number looks better, and no stable advantage survives
#           on a period the search never touched. Whether B happens to finish
#           slightly ahead or behind on one fresh realization is random. The
#           files do not record the amount of search; the research record must.

### Demo 5 — Draw label intervals and compare validation clocks

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
rng = np.random.default_rng(5)
T = 1800
horizon = 10
state = np.zeros(T)
innovation = rng.standard_normal(T)
for t in range(1, T):
    state[t] = 0.98 * state[t - 1] + innovation[t]

one_period_outcome = 0.06 * state + rng.standard_normal(T)
future = pd.concat(
    [
        pd.Series(one_period_outcome).shift(-step)
        for step in range(1, horizon + 1)
    ],
    axis=1,
)
overlap_y = future.mean(axis=1, skipna=False).to_numpy()
valid_rows = np.flatnonzero(np.isfinite(overlap_y))
validation_X = state[valid_rows, None]
validation_y = overlap_y[valid_rows]
label_start = valid_rows + 1
label_end = valid_rows + horizon


def regression_fold_score(train_rows, test_rows):
    model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10))
    model.fit(validation_X[train_rows], validation_y[train_rows])
    prediction = model.predict(validation_X[test_rows])
    denominator = np.sum(
        (validation_y[test_rows] - validation_y[train_rows].mean()) ** 2
    )
    return 1 - np.sum((validation_y[test_rows] - prediction) ** 2) / denominator


n_valid = len(valid_rows)
fold_size = 240
starts = [600, 840, 1080, 1320]
forward_scores = []
purged_scores = []
purged_counts = []
interval_rows = []
for start in starts:
    stop = min(start + fold_size, n_valid)
    test_rows = np.arange(start, stop)
    train_rows = np.arange(0, start)
    assessment_start = label_start[test_rows].min()
    assessment_end = label_end[test_rows].max()
    keep = label_end[train_rows] < assessment_start
    purged_train = train_rows[keep]

    forward_scores.append(regression_fold_score(train_rows, test_rows))
    purged_scores.append(regression_fold_score(purged_train, test_rows))
    purged_counts.append(len(train_rows) - len(purged_train))
    interval_rows.append(
        {
            "test_rows": f"{test_rows[0]}:{test_rows[-1]}",
            "assessment_label_interval": f"{assessment_start}:{assessment_end}",
            "last_training_label_end": int(label_end[train_rows[-1]]),
            "purged_rows": int(len(train_rows) - len(purged_train)),
        }
    )

shuffled_scores = []
for train_rows, test_rows in KFold(
    n_splits=5, shuffle=True, random_state=0
).split(validation_X):
    shuffled_scores.append(regression_fold_score(train_rows, test_rows))

print("\nFirst four forward assessment boundaries")
print(pd.DataFrame(interval_rows).to_string(index=False))
first_start = starts[0]
first_assessment_start = int(label_start[first_start])
first_train_rows = np.arange(0, first_start)
first_keep = label_end[first_train_rows] < first_assessment_start
last_kept = int(first_train_rows[first_keep][-1])
first_purged = int(first_train_rows[~first_keep][0])
last_nominal = int(first_train_rows[-1])
boundary_examples = pd.DataFrame([
    {
        "role": "last eligible training row",
        "row": last_kept,
        "label_interval": f"{label_start[last_kept]}:{label_end[last_kept]}",
    },
    {
        "role": "first purged training row",
        "row": first_purged,
        "label_interval": f"{label_start[first_purged]}:{label_end[first_purged]}",
    },
    {
        "role": "last nominal training row",
        "row": last_nominal,
        "label_interval": f"{label_start[last_nominal]}:{label_end[last_nominal]}",
    },
])
print(f"\nFirst assessment outcome interval begins at {first_assessment_start}")
print(boundary_examples.to_string(index=False))
print("\nMean validation R2")
print(f"Shuffled historical mixture: {np.mean(shuffled_scores):+.4f}")
print(f"Forward without purge:       {np.mean(forward_scores):+.4f}")
print(f"Purged forward:              {np.mean(purged_scores):+.4f}")
print(f"Rows purged per fold:         {purged_counts}")
# Expected: exact purging removes nine crossing label intervals. Shuffled and
# forward designs answer different chronology questions; neither must be larger.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| label and first assessment boundary | `[label horizon and first assessment row]` |
| last eligible training row | `[row and full label interval]` |
| first purged training row | `[row and full label interval]` |
| last nominal training row | `[row and full label interval]` |
| exact purge count | `[number removed at each boundary]` |
| mean splitter results | `[shuffled, forward, and purged-forward R²]` |
| mechanism | `[why overlapping labels cross the boundary]` |
| narrow claim | `[what the splitter comparison demonstrates]` |
| limitation | `[why the three scores need not have a fixed ordering]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed boundary-audit entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| label and first assessment boundary | each label averages the next 10 outcomes; first assessment rows 600–839 have outcome interval 601–849 |
| last eligible training row | row **590**, label interval **591–600**; its label ends before assessment outcomes begin |
| first purged training row | row **591**, label interval **592–601**; it shares outcome 601 with assessment |
| last nominal training row | row **599**, label interval **600–609** |
| exact purge count | **9 rows** in every forward fold |
| mean splitter results | shuffled historical mixture R² **+0.3417**; forward without purge **+0.2343**; purged forward **+0.2453** |
| mechanism | shuffled versus forward changes chronology and historical mixture; forward versus purged changes only the crossing label intervals near each boundary |
| narrow claim | exact interval arithmetic identified and removed this overlap mechanism; the score ordering itself is not a universal bias formula |
| limitation | purging does not certify feature vintages, preprocessing, universe history, shifts, independence, or the broader model search |

Purging can move a score in either direction. The construction tells us which
information conflict was removed; the sign of the score change does not.

</details>

### Demo 6 — Three production clocks

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
dates = pd.date_range("2024-01-31", periods=12, freq="ME")
series_clock = pd.DataFrame(
    {
        "date": dates,
        "fold": ["train"] * 6 + ["validation"] * 3 + ["test"] * 3,
    }
)

panel_clock = pd.MultiIndex.from_product(
    [dates[-6:], ["A", "B", "C", "D"]], names=["date", "asset"]
).to_frame(index=False)
panel_clock["fold"] = np.where(
    panel_clock["date"] < dates[-2], "train", "assessment"
)

events = pd.DataFrame(
    {
        "company": ["A", "A", "B", "B", "C", "C"],
        "filing_family": ["A-2024", "A-2024", "B-2024", "B-2024", "C-2024", "C-2024"],
        "edition": ["original", "amendment"] * 3,
        "period_end": pd.to_datetime(
            ["2024-12-31", "2024-12-31"] * 3
        ),
        "public_release": pd.to_datetime(
            [
                "2025-02-15",
                "2025-03-01",
                "2025-02-20",
                "2025-03-05",
                "2025-04-10",
                "2025-04-18",
            ]
        ),
    }
)
# Assign the complete related filing family according to its first public release.
family_release = events.groupby("filing_family")["public_release"].transform("min")
events["fold"] = np.where(family_release < pd.Timestamp("2025-04-01"), "train", "assessment")

print("\nSingle-series clock")
print(series_clock.to_string(index=False))
assert series_clock["fold"].tolist() == ["train"] * 6 + ["validation"] * 3 + ["test"] * 3
print("\nPanel clock: no date is split")
print(panel_clock.groupby(["date", "fold"]).size().rename("rows").to_string())
assert panel_clock.groupby("date")["fold"].nunique().max() == 1
print("\nEvent/document clock: related editions stay together")
print(events.to_string(index=False))
assert events.groupby("filing_family")["fold"].nunique().max() == 1
print("Clock assertions passed: series order, intact panel dates, and intact filing families.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| single-series unit | `[protected block and required ordering]` |
| panel unit | `[protected block and required grouping]` |
| event/document unit | `[protected family and grouping key]` |
| event time rule | `[ordering or embargo needed for the application]` |
| executable checks | `[assertions run and result]` |
| narrow claim | `[what the three examples establish]` |
| limitation | `[what domain-specific dependence remains to be specified]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed production-clock entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| single-series unit | one dated monthly observation; January–June train, July–September validation, October–December test |
| panel unit | the complete four-asset cross-section for each month; no date appears in more than one fold |
| event/document unit | one related filing family; A and B editions remain in training, while both C editions remain in assessment |
| event time rule | use public release time, not fiscal period end; group original and amendment before assigning the fold |
| executable checks | fold order matches the declared series clock; maximum folds per panel date = 1; maximum folds per filing family = 1 |
| narrow claim | the three tables implement three different protected units rather than applying one universal row splitter |
| limitation | these toy tables do not decide whether deployment should generalize across familiar or unseen entities; that claim must be declared first |

The correct split follows what arrives together and what the future system is
being asked to generalize to.

</details>

### Demo 7 — Nested selection exposes the difference between choosing and assessing

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
rng = np.random.default_rng(7)
n = 1350
n_features = 120
search_X = rng.standard_normal((n, n_features))
search_y = rng.standard_normal(n)

# Prespecify candidate feature sets before looking at any fold outcome.
candidate_sets = [
    rng.choice(n_features, size=6, replace=False)
    for _ in range(50)
]
outer_rows = []
for outer_start in [600, 750, 900, 1050, 1200]:
    outer_stop = min(outer_start + 150, n)
    outer_test = np.arange(outer_start, outer_stop)
    development = np.arange(0, outer_start)
    inner_validation = development[-150:]
    inner_train = development[:-150]

    candidate_losses = []
    for columns in candidate_sets:
        candidate = Ridge(alpha=10.0).fit(
            search_X[inner_train][:, columns], search_y[inner_train]
        )
        candidate_losses.append(
            mse(
                search_y[inner_validation],
                candidate.predict(search_X[inner_validation][:, columns]),
            )
        )

    winner = int(np.argmin(candidate_losses))
    selected_columns = candidate_sets[winner]
    frozen_model = Ridge(alpha=10.0).fit(
        search_X[development][:, selected_columns], search_y[development]
    )
    outer_loss = mse(
        search_y[outer_test],
        frozen_model.predict(search_X[outer_test][:, selected_columns]),
    )
    baseline_forecast = np.repeat(search_y[development].mean(), len(outer_test))
    baseline_loss = mse(search_y[outer_test], baseline_forecast)
    outer_rows.append(
        {
            "outer_period": f"{outer_test[0]}:{outer_test[-1]}",
            "selected_candidate": winner,
            "best_inner_loss": candidate_losses[winner],
            "outer_assessed_loss": outer_loss,
            "outer_baseline_loss": baseline_loss,
        }
    )

nested_results = pd.DataFrame(outer_rows)
print("\nNested selection among 50 equally useless candidates")
print(nested_results.round(4).to_string(index=False))
print(
    "Mean best inner loss, outer assessed loss, and outer baseline loss: "
    f"{nested_results['best_inner_loss'].mean():.4f} vs "
    f"{nested_results['outer_assessed_loss'].mean():.4f} vs "
    f"{nested_results['outer_baseline_loss'].mean():.4f}"
)

fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(len(nested_results))
ax.plot(x, nested_results["best_inner_loss"], "o-", label="selected inner loss")
ax.plot(x, nested_results["outer_assessed_loss"], "s-", label="outer assessed loss")
ax.set(
    xlabel="outer forecast origin",
    ylabel="mean squared loss",
    xticks=x,
    xticklabels=nested_results["outer_period"],
)
ax.tick_params(axis="x", rotation=25)
ax.legend()
fig.tight_layout()
plt.show()
# Expected: the inner selected loss is better than the outer assessed loss. That gap is
#           what the search cost.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| known truth and search | `[planted signal and number of candidate features]` |
| outer design | `[number, size, and chronology of assessment blocks]` |
| inner rule | `[how one candidate is selected without touching the outer block]` |
| period evidence | `[selected candidate, inner loss, outer loss, and baseline loss by period]` |
| aggregate evidence | `[mean inner, outer, and frozen-baseline loss]` |
| narrow claim | `[what the gap between selection and assessment shows]` |
| limitation | `[what this small simulation cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed nested-selection entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| known truth and search | 1,350 synthetic rows, 120 noise features, noise target, and 50 prespecified six-feature Ridge candidates; all candidates are useless |
| outer design | five expanding development samples with successive 150-row outer assessment blocks from rows 600–1349 |
| inner rule | the final 150 eligible development rows select one of 50 candidates; the winner is refitted on all outer development history |
| period evidence | selected candidates differ by origin; outer assessed loss is sometimes better and sometimes worse than selected inner loss |
| aggregate evidence | mean best inner loss **0.9615**; mean outer assessed loss **0.9713**; mean frozen development-mean baseline loss **0.9701** |
| narrow claim | selection made inner loss look slightly better, while the full selected procedure did not beat the feasible baseline on average outside |
| limitation | one noise design, one inner holdout, fixed Ridge penalty, and overlapping expanding histories do not quantify a universal search penalty or provide iid uncertainty |

The headline is the outer procedure-versus-baseline comparison. The selected
inner loss records how the winner was chosen, not how well it traveled.

</details>

## Where this leaves us

The same probability model can support different decisions because costs, benefits, constraints, and thresholds belong to the policy layer. A good ranking can coexist with poor probability estimates, and a carefully chosen policy still needs genuinely later assessment.

Week 5 increases model flexibility with trees and ensembles, then asks what that extra representational power can learn—and what extra evidence it must earn.